In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
# imports
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.feature_engineering import (
    BureauMergerAndImputer,
    process_bureau_data,
    DropHighMissingColumns,
    AddMissingIndicators,
    ApplicationFeatureEngineer,
    FrequencyEncoder,
    HighCorrelationFilter,
)

In [3]:
# Load Data & Train/Test Split
data_dir = Path("../data")
df_app = pd.read_csv(data_dir / "application_train.csv")
df_bureau = pd.read_csv(data_dir / "bureau.csv")

df_app_train, df_app_test, y_train, y_test = train_test_split(df_app.iloc[:, :-1], df_app.iloc[:, -1], test_size=0.2, random_state=42)

In [4]:
# Initialize the Pipeline with the custom transformer
bur_pipeline = Pipeline([
    ("bureau_merger", BureauMergerAndImputer(df_bureau=df_bureau)),
])

# pipeline that merges the aggregated bureau dataframe with the application data
bur_pipeline.fit(df_app_train, y_train)

# merging & transforming the datasets via the pipeline
df_train = bur_pipeline.transform(df_app_train)
df_test = bur_pipeline.transform(df_app_test)

In [5]:
# feature engineering pipeline

fe_pipeline = Pipeline(
    steps=[
        ("drop_high_missing", DropHighMissingColumns(threshold=65.0)),          # dropping columns with more than 65% missing
        ("add_missing_indicators", AddMissingIndicators(threshold=5.0)),        # adding missing indicator columns where missing percentage is > 5%
        ("feature_engineer", ApplicationFeatureEngineer()),                     # feature engineering the merged dataframe
        ("remove_correlated_features", HighCorrelationFilter(threshold=0.95)),  # removing one of the features where correlation score is more than 95%
        ("frequency_encoding", FrequencyEncoder(cols=["OCCUPATION_TYPE"]))      # frequency encoding the 'OCCUPATION_TYPE' column
    ]
)

df_train_fe = fe_pipeline.fit_transform(df_train)
df_test_fe = fe_pipeline.transform(df_test)

### Justification for Data Preprocessing and Feature Engineering

- **Missing-value handling:** Columns with more than 65% missing values were removed because they contain insufficient information to reliably contribute to the model. Missing-value indicators were added for features with at least 5% missingness to preserve potentially predictive information contained in the pattern of missing data.

- **Feature engineering:** New features were created to capture financially meaningful relationships that are not directly represented by the raw variables. These features summarize applicants' affordability, repayment burden, employment characteristics, household income, external credit information, and age-related patterns.

- **Bureau aggregation features:** Aggregations from `bureau.csv` were created to incorporate applicants' historical credit behavior into the application-level dataset. Counts, credit activity, credit recency, and overdue amounts provide a broader view of an applicant's existing and historical credit exposure.

- **Correlation-based feature filtering:** Pearson correlation was used to identify and remove features with a correlation of 0.95 or higher. This reduces redundant information, limits multicollinearity, and produces a more efficient feature set without retaining highly similar variables.

- **Frequency encoding:** `OCCUPATION_TYPE` was frequency encoded because it is a categorical variable with multiple categories and one-hot encoding would increase the dimensionality of the dataset. Frequency encoding retains information about the prevalence of each occupation while representing the variable as a single numerical feature.